# Result Statistics

统计某个结果目录下所有运行的平均指标：
- 平均 `accuracy`
- 平均 `harness_metrics`（逐字段）
- 平均实例通过率（对每个 result 的 `instance_results` 先算通过率，再求平均）


In [1]:
from __future__ import annotations

import json
from pathlib import Path
from statistics import mean
from typing import Any

HARNESS_METRIC_KEYS = [
    "total_input_tokens",
    "total_output_tokens",
    "total_cache_tokens",
    "total_cost",
    "average_cost",
    "average_steps",
    "average_steps_passed",
    "instances_with_metrics",
    "denominator_instances",
    "denominator_passed_instances",
]


def _to_float(value: Any, default: float = 0.0) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def _find_eval_files(results_root: str | Path) -> list[Path]:
    root = Path(results_root).expanduser()
    candidates = [root]
    if not root.is_absolute():
        candidates.append(Path("harness") / root)

    for base in candidates:
        if not base.exists():
            continue

        direct = base / "eval_results.json"
        if direct.is_file():
            return [direct]

        child = sorted(base.glob("*/eval_results.json"))
        if child:
            return child

        recursive = sorted(base.rglob("eval_results.json"))
        if recursive:
            return recursive

    checked = ", ".join(str(x) for x in candidates)
    raise FileNotFoundError(f"No eval_results.json found. Checked: {checked}")


def _iter_payloads(eval_file: Path):
    data = json.loads(eval_file.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        return
    for agent_name, payload in data.items():
        if isinstance(payload, dict):
            yield agent_name, payload


def summarize_results(results_root: str | Path) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    eval_files = _find_eval_files(results_root)
    rows: list[dict[str, Any]] = []

    for eval_file in eval_files:
        for agent_name, payload in _iter_payloads(eval_file):
            instance_results = payload.get("instance_results", {})
            if isinstance(instance_results, dict) and instance_results:
                passed = sum(1 for v in instance_results.values() if bool(v))
                total = len(instance_results)
                instance_pass_rate = passed / total
            else:
                passed = 0
                total = 0
                instance_pass_rate = 0.0

            harness_metrics = payload.get("harness_metrics", {})
            if not isinstance(harness_metrics, dict):
                harness_metrics = {}

            row = {
                "run_name": eval_file.parent.name,
                "agent_name": agent_name,
                "source_file": str(eval_file),
                "accuracy": _to_float(payload.get("accuracy", 0.0)),
                "instance_pass_rate": instance_pass_rate,
                "instance_passed": passed,
                "instance_total": total,
            }
            for key in HARNESS_METRIC_KEYS:
                row[key] = _to_float(harness_metrics.get(key, 0.0))
            rows.append(row)

    if not rows:
        raise ValueError(f"No valid agent payload found under: {results_root}")

    summary = {
        "results_root": str(results_root),
        "num_eval_files": len(eval_files),
        "num_results": len(rows),
        "average_accuracy": mean(r["accuracy"] for r in rows),
        "average_instance_pass_rate": mean(r["instance_pass_rate"] for r in rows),
        "average_harness_metrics": {
            key: mean(r[key] for r in rows)
            for key in HARNESS_METRIC_KEYS
        },
    }
    return summary, rows


def print_summary(summary: dict[str, Any]) -> None:
    print(f"results_root: {summary['results_root']}")
    print(f"num_eval_files: {summary['num_eval_files']}")
    print(f"num_results: {summary['num_results']}")
    print(f"average_accuracy: {summary['average_accuracy']:.6f}")
    print(f"average_instance_pass_rate: {summary['average_instance_pass_rate']:.6f}")
    print()
    print("Average harness_metrics:")
    for key, value in summary["average_harness_metrics"].items():
        print(f"  {key}: {value:.6f}")


def print_rows(rows: list[dict[str, Any]], max_rows: int = 20) -> None:
    print()
    print("Per-result values (showing first rows):")
    header = f"{'run_name':<24} {'agent_name':<32} {'accuracy':>10} {'pass_rate':>10}"
    print(header)
    print("-" * len(header))
    for row in rows[:max_rows]:
        print(
            f"{row['run_name']:<24} {row['agent_name']:<32} "
            f"{row['accuracy']:>10.6f} {row['instance_pass_rate']:>10.6f}"
        )
    if len(rows) > max_rows:
        print(f"... ({len(rows) - max_rows} more rows)")


In [2]:
# 示例：统计 results/baseline 下各次运行的平均值
summary, rows = summarize_results("results/baseline")
print_summary(summary)
print_rows(rows)


results_root: results/baseline
num_eval_files: 10
num_results: 10
average_accuracy: 0.430500
average_instance_pass_rate: 0.430500

Average harness_metrics:
  total_input_tokens: 1517612991.600000
  total_output_tokens: 5350670.600000
  total_cache_tokens: 1502984828.800000
  total_cost: 182.741725
  average_cost: 0.913709
  average_steps: 167.801000
  average_steps_passed: 160.794551
  instances_with_metrics: 200.000000
  denominator_instances: 200.000000
  denominator_passed_instances: 86.100000

Per-result values (showing first rows):
run_name                 agent_name                         accuracy  pass_rate
-------------------------------------------------------------------------------
baseline_v2_run1         mini_baseline                      0.410000   0.410000
baseline_v2_run10        minisweagent-v2-20260221160318-aac6   0.400000   0.400000
baseline_v2_run2         minisweagent-v2-20260217124019-8d39   0.450000   0.450000
baseline_v2_run3         minisweagent-v2-2026021814

In [3]:
# 示例：统计 results/baseline 下各次运行的平均值
summary, rows = summarize_results("results/toolcreate")
print_summary(summary)
print_rows(rows)

results_root: results/toolcreate
num_eval_files: 3
num_results: 3
average_accuracy: 0.408333
average_instance_pass_rate: 0.408333

Average harness_metrics:
  total_input_tokens: 1313996827.000000
  total_output_tokens: 4801854.000000
  total_cache_tokens: 1300247904.000000
  total_cost: 159.286242
  average_cost: 0.796431
  average_steps: 146.166667
  average_steps_passed: 141.846379
  instances_with_metrics: 200.000000
  denominator_instances: 200.000000
  denominator_passed_instances: 81.666667

Per-result values (showing first rows):
run_name                 agent_name                         accuracy  pass_rate
-------------------------------------------------------------------------------
run1                     toolcreate-20260223002645-236d     0.430000   0.430000
run2                     toolcreate-20260223002656-70e2     0.380000   0.380000
run3                     toolcreate-20260223002714-87bd     0.415000   0.415000


In [6]:
# 示例：统计 results/baseline 下各次运行的平均值
summary, rows = summarize_results("results/agentic-mem")
print_summary(summary)
print_rows(rows)

results_root: results/agentic-mem
num_eval_files: 4
num_results: 4
average_accuracy: 0.433750
average_instance_pass_rate: 0.433750

Average harness_metrics:
  total_input_tokens: 1331832165.250000
  total_output_tokens: 4622559.500000
  total_cache_tokens: 1317531636.000000
  total_cost: 160.694156
  average_cost: 0.803471
  average_steps: 141.792500
  average_steps_passed: 137.634723
  instances_with_metrics: 200.000000
  denominator_instances: 200.000000
  denominator_passed_instances: 86.750000

Per-result values (showing first rows):
run_name                 agent_name                         accuracy  pass_rate
-------------------------------------------------------------------------------
run1                     agentic-mem-20260223010523-4ad7    0.435000   0.435000
run2                     agentic-mem-20260223010538-2a28    0.450000   0.450000
run3                     agentic-mem-20260223010557-73d5    0.445000   0.445000
run4                     agentic-mem-20260223145855-504b

In [5]:
# 示例：统计 results/baseline 下各次运行的平均值
summary, rows = summarize_results("results/externalmem")
print_summary(summary)
print_rows(rows)

results_root: results/externalmem
num_eval_files: 6
num_results: 6
average_accuracy: 0.445000
average_instance_pass_rate: 0.445000

Average harness_metrics:
  total_input_tokens: 1491537753.500000
  total_output_tokens: 5298952.000000
  total_cache_tokens: 1474905048.000000
  total_cost: 181.570694
  average_cost: 0.907853
  average_steps: 163.244167
  average_steps_passed: 155.571334
  instances_with_metrics: 200.000000
  denominator_instances: 200.000000
  denominator_passed_instances: 89.000000

Per-result values (showing first rows):
run_name                 agent_name                         accuracy  pass_rate
-------------------------------------------------------------------------------
run1                     external-mem-20260221002153-1b3e   0.420000   0.420000
run2                     external-mem-20260221002213-80a6   0.435000   0.435000
run3                     external-mem-20260221002259-4bce   0.440000   0.440000
run4                     external-mem-20260221160343-05f